# RideShare — demand, earnings and cancellations
Python DB-API (`psycopg2`) + pandas + matplotlib on the PostgreSQL `rideshare` database.
Set `PGHOST`, `PGPORT`, `PGUSER`, `PGPASSWORD`, `PGDATABASE` if your server is not on `localhost:5432`.

In [1]:
import os, psycopg2, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
os.makedirs("docs", exist_ok=True)
conn = psycopg2.connect(host=os.getenv("PGHOST","localhost"), port=os.getenv("PGPORT","5432"),
                        user=os.getenv("PGUSER","postgres"), password=os.getenv("PGPASSWORD",""),
                        dbname=os.getenv("PGDATABASE","rideshare"))
def run(sql, params=None):
    """DB-API cursor -> DataFrame (parameterised, so no string-built SQL)."""
    with conn.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
run("SELECT COUNT(*) AS trips, COUNT(*) FILTER (WHERE status='COMPLETED') AS completed FROM trip")

,trips,completed
0,6500,5276


## 1. Demand by hour of day

In [2]:
hourly = run("""SELECT EXTRACT(HOUR FROM requested_at)::int AS hour, COUNT(*) AS requests
                 FROM trip GROUP BY 1 ORDER BY 1""")
peak = hourly.requests >= hourly.requests.quantile(0.75)
fig, ax = plt.subplots(figsize=(6.4,4.2), dpi=160)
ax.bar(hourly.hour, hourly.requests, color=["#0F6E6E" if p else "#B9D3D3" for p in peak])
ax.set_xticks(range(0,24,3)); ax.set_xlabel("Hour of day"); ax.set_ylabel("Trip requests")
ax.set_title("Requests peak at 8-9 am and 5-7 pm", loc="left", fontsize=11, fontweight="bold")
for s in ("top","right"): ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig("docs/chart_hourly.png"); hourly.sort_values("requests", ascending=False).head(5)

,hour,requests
18,18,548
19,19,547
17,17,530
8,8,493
9,9,439


## 2. Demand by pickup zone and time band

In [3]:
zb = run("""SELECT z.zone_name,
   CASE WHEN h BETWEEN 6 AND 9 THEN 'Morning (6-9)' WHEN h BETWEEN 10 AND 16 THEN 'Midday (10-16)'
        WHEN h BETWEEN 17 AND 20 THEN 'Evening (17-20)' ELSE 'Night (21-5)' END AS band, COUNT(*) AS n
 FROM (SELECT pickup_zone, EXTRACT(HOUR FROM requested_at)::int AS h FROM trip) t
 JOIN zone z ON z.zone_id=t.pickup_zone GROUP BY 1,2""")
pv = zb.pivot(index="zone_name", columns="band", values="n")[["Morning (6-9)","Midday (10-16)","Evening (17-20)","Night (21-5)"]]
pv = pv.loc[pv.sum(axis=1).sort_values().index]
cols = ["#7FB5B5","#B9D3D3","#0F6E6E","#E3A857"]
fig, ax = plt.subplots(figsize=(6.4,4.2), dpi=160)
left = None
import numpy as np
left = np.zeros(len(pv))
for c,col in zip(pv.columns, cols):
    ax.barh(pv.index, pv[c], left=left, color=col, label=c); left += pv[c].values
ax.set_xlabel("Trip requests"); ax.legend(frameon=False, fontsize=8, ncol=4, loc="upper center", bbox_to_anchor=(0.45,-0.16), columnspacing=1)
ax.set_title("MG Road and Koramangala lead; Whitefield peaks in the evening", loc="left", fontsize=10.5, fontweight="bold")
for s in ("top","right"): ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig("docs/chart_zone_band.png"); pv

band,Morning (6-9),Midday (10-16),Evening (17-20),Night (21-5)
zone_name,,,,
Hebbal,176,201,115,92
Jayanagar,181,220,130,104
Electronic City,105,179,298,89
Indiranagar,140,305,204,140
HSR Layout,229,271,184,107
Whitefield,151,243,386,124
Koramangala,193,365,250,188
MG Road,203,326,453,148


## 3. Driver earnings and ratings

In [4]:
dp = run("SELECT * FROM v_driver_performance ORDER BY gross_earnings DESC")
top = dp.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(6.4,4.2), dpi=160)
ax.barh(top.full_name, top.gross_earnings.astype(float)/1000, color="#0F6E6E")
for y,(e,r) in enumerate(zip(top.gross_earnings.astype(float)/1000, top.avg_rating)): ax.text(e+0.6, y, f"★{r}", va="center", fontsize=8)
ax.set_xlabel("Gross earnings (₹ thousand)"); ax.set_title("Top 10 drivers by earnings", loc="left", fontsize=11, fontweight="bold")
for s in ("top","right"): ax.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig("docs/chart_drivers.png")
dp[["completed_trips","gross_earnings","avg_rating"]].astype(float).corr().round(2)

,completed_trips,gross_earnings,avg_rating
completed_trips,1.00,0.83,0.01
gross_earnings,0.83,1.00,-0.03
avg_rating,0.01,-0.03,1.00


## 4. Cancellations: zone and driver view

In [5]:
cz = run("""SELECT z.zone_name, ROUND(100.0*COUNT(*) FILTER (WHERE status='CANCELLED')/COUNT(*),1) AS cancel_pct
            FROM trip t JOIN zone z ON z.zone_id=t.pickup_zone GROUP BY 1 ORDER BY 2 DESC""")
dc = run("""SELECT driver_id, COUNT(*) AS accepted,
            100.0*COUNT(*) FILTER (WHERE cancelled_by='DRIVER')/COUNT(*) AS cancel_pct, AVG(rider_rating) AS rating
            FROM trip WHERE driver_id IS NOT NULL GROUP BY 1""").astype(float)
fig, ax = plt.subplots(1,2,figsize=(10,4), dpi=160)
ax[0].barh(cz.zone_name[::-1], cz.cancel_pct.astype(float)[::-1], color="#E3A857"); ax[0].set_xlabel("% of requests cancelled")
ax[0].set_title("Cancellation rate by pickup zone", loc="left", fontsize=10.5, fontweight="bold")
ax[1].scatter(dc.rating, dc.cancel_pct, s=dc.accepted/3, color="#0F6E6E", alpha=.7)
ax[1].set_xlabel("Average rider rating"); ax[1].set_ylabel("% cancelled by driver after accepting")
ax[1].set_title("Drivers: rating vs cancellation", loc="left", fontsize=10.5, fontweight="bold")
for a in ax:
    for s in ("top","right"): a.spines[s].set_visible(False)
fig.tight_layout(); fig.savefig("docs/chart_cancel.png")
print("overall cancellation %:", run("SELECT ROUND(100.0*COUNT(*) FILTER (WHERE status='CANCELLED')/COUNT(*),1) AS p FROM trip").p[0])
cz

overall cancellation %: 18.8


,zone_name,cancel_pct
0,Whitefield,21.1
1,Jayanagar,20.0
2,Koramangala,19.5
3,MG Road,19.2
4,Hebbal,18.3
5,HSR Layout,18.0
6,Indiranagar,17.2
7,Electronic City,16.4


In [6]:
conn.close()